### Pipeline Run Modes
The notebook supports two run patterns controlled by `PIPELINE_MODE`:
- `incremental`: default behavior. Appends only new work orders based on the most recent delta between `fleet_ai_master` and `fleet_ai`.
- `reprocess`: rebuilds a specified date window end-to-end, upserting into both `fleet_ai_master` and `fleet_ai`.
Usage:
1. Set `PIPELINE_MODE` to either `"incremental"` or `"reprocess"`.
2. Optionally override `PIPELINE_START_DATE` (default `2022-01-01`) and, for reprocess runs, `PIPELINE_END_DATE`.
3. Run the notebook; `run_pipeline()` picks up the configured mode and date range automatically.
#### Notebook Overview
This notebook performs an end-to-end Fleet AI pipeline. High level:
1) Extract: pull raw Work Order headers since 2022-01-01 from `Work_Order_Header`.
2) Enrich: join lookup dimensions (equipment, manufacturer, service type).
3) Comments: flatten and attach Complaint/Cause/Cure/Int comments.
4) Parts: aggregate parts used and total amount invoiced.
5) Persist: append only new rows to lakehouse Delta table `fleet_ai_master`.
6) Select for AI: find work orders present in `fleet_ai_master` but missing from the Lakehouse table; store as `pending_ai_records_df`.
7) AI: generate Service_Category, Damage_Category, and a customer-facing summary comment.
8) Publish: append only new AI-enriched rows to the Lakehouse table.
Key intermediate outputs:
- `final_df`: enriched work orders (facts) written to `fleet_ai_master`.
- pending_ai_records_df: subset that still needs AI processing + publishing.
- `pandas_df` -> `spark_df`: AI-enriched results prepared for Synapse.


In [1]:
from typing import Dict, Iterable, Optional, Tuple

import pyspark.sql.functions as F
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.functions import broadcast
from pyspark.sql.types import StringType, StructField, StructType
from pyspark.sql.utils import AnalysisException

APP_NAME = "Combined_WorkOrder_Comments"
DEFAULT_START_DATE = "2026-03-04"
DEFAULT_LAKEHOUSE = "DE_LH_SILVER"
DELTA_TARGET_TABLE = f"{DEFAULT_LAKEHOUSE}.fleet_ai_master"
KEY_COLUMN = "Work_Order_No"
TARGET_TABLE_NAME = "fleet_ai"
LAKEHOUSE_TARGET_TABLE = f"{DEFAULT_LAKEHOUSE}.{TARGET_TABLE_NAME}"
INCREMENTAL_MODE = "incremental"
REPROCESS_MODE = "reprocess"

# Runtime-configurable pipeline switches. Update PIPELINE_MODE to "reprocess" and provide
# PIPELINE_START_DATE / PIPELINE_END_DATE (inclusive) when rerunning a historic window.
PIPELINE_MODE = REPROCESS_MODE #changed 4/7/2026 for prompt change
PIPELINE_START_DATE = DEFAULT_START_DATE
PIPELINE_END_DATE: Optional[str] = "2026-03-07" # or set to `None`

SYNAPSE_OUTPUT_COLUMNS = [
    "Work_Order_No",
    "Starting_Date",
    "Sell_to_Customer_No",
    "Responsibility_Center",
    "Service_Type_Description",
    "Equipment_Description",
    "Equipment_Category",
    "Equipment_Group",
    "Equipment_Model",
    "Manufacturer",
    "Year",
    "Key_Hours",
    "Pump_Hours",
    "Drive_Hours",
    "Complaint",
    "Cause",
    "Cure_Comment",
    "Int_Comment",
    "Parts_Used_Full",
    "Service_Category",
    "Damage_Category",
    "Work_Order_Comment",
]
SYNAPSE_COLUMN_CASTS = {
    "Work_Order_No": "string",
    "Starting_Date": "timestamp",
    "Sell_to_Customer_No": "string",
    "Responsibility_Center": "long",
    "Service_Type_Description": "string",
    "Equipment_Description": "string",
    "Equipment_Category": "string",
    "Equipment_Group": "string",
    "Equipment_Model": "string",
    "Manufacturer": "string",
    "Year": "long",
    "Key_Hours": "double",
    "Pump_Hours": "long",
    "Drive_Hours": "long",
    "Complaint": "string",
    "Cause": "string",
    "Cure_Comment": "string",
    "Int_Comment": "string",
    "Parts_Used_Full": "string",
    "Service_Category": "string",
    "Damage_Category": "string",
    "Work_Order_Comment": "string",
}

 


def lakehouse_table(table_name: str) -> str:
    """Return the fully qualified table path within the default lakehouse."""
    return f"{DEFAULT_LAKEHOUSE}.{table_name}"


def read_lakehouse_table(spark: SparkSession, table_name: str) -> DataFrame:
    """Read a table from the default lakehouse using the canonical name."""
    return spark.read.table(lakehouse_table(table_name))


LOOKUP_DEFINITIONS: Dict[str, Tuple[str, Iterable[str]]] = {
    "Equipment_Object": (
        "Equipment_Object",
        (
            "No_",
            "Equipment_Category",
            "Equipment_Group",
            "Equipment_Model",
            "Manufacturer_Code",
            "Description",
            "Year",
        ),
    ),
    "Equipment_Category": ("Equipment_Category", ("Code", "Description")),
    "Equipment_Group": ("Equipment_Group", ("Code", "Description")),
    "Equipment_Model": ("Equipment_Model", ("No_", "Description")),
    "Manufacturer": ("Manufacturer", ("code", "name")),
    "Service_Type": (
        "Service_Type",
        ("Code", "Description", "Service_Description"),
    ),
    "Work_Order_Line": (
        "Work_Order_Line",
        ("Document_No_", "No_", "Amount_Invoiced"),
    ),
    "Item": (
        "Item",
        ("No_", "Description", "Manufacturer_Code", "Item_Category_Code"),
    ),
    "Item_Category": ("Item_Category", ("Code", "Description")),
}

# ---------- Spark init ----------
def init_spark(app_name: str = APP_NAME) -> SparkSession:
    """Return the active Spark session or initialize a new one with required configs."""
    parquet_rebase_configs = {
        "spark.sql.parquet.int96RebaseModeInWrite": "LEGACY",
        "spark.sql.parquet.datetimeRebaseModeInWrite": "LEGACY",
        "spark.sql.parquet.int96RebaseModeInRead": "LEGACY",
        "spark.sql.parquet.datetimeRebaseModeInRead": "LEGACY",
    }

    active = SparkSession.getActiveSession()
    if active:
        # Ensure the active session has the expected parquet rebase behavior.
        for key, value in parquet_rebase_configs.items():
            active.conf.set(key, value)
        return active

    builder = SparkSession.builder.appName(app_name)
    for key, value in parquet_rebase_configs.items():
        builder = builder.config(key, value)

    return builder.getOrCreate()


# ---------- Dimension look-ups ----------
def load_lookup_tables(spark: SparkSession) -> Dict[str, DataFrame]:
    """Load all lookup tables defined in LOOKUP_DEFINITIONS."""
    lookups: Dict[str, DataFrame] = {}
    for alias, (table_name, columns) in LOOKUP_DEFINITIONS.items():
        lookups[alias] = read_lakehouse_table(spark, table_name).select(*columns)
    return lookups

# ---------- Core work-order pull ----------
def prepare_work_orders(
    spark: SparkSession,
    start_date: str = DEFAULT_START_DATE,
    end_date: Optional[str] = None,
) -> DataFrame:
    """
    Return work orders posted from start_date (inclusive) through end_date (inclusive)
    with 'Document_Type = 1' and 'Posting_Status = 2'. When end_date is None the upper
    bound defaults to the current date.
    """
    start_bound = F.to_date(F.lit(start_date))
    end_bound = F.to_date(F.lit(end_date)) if end_date else F.current_date()

    return (
        read_lakehouse_table(spark, "Work_Order_Header")
        .filter(
            (F.col("Posting_Status") == 2)
            & (F.col("Document_Type") == 1)
            & (F.col("Posting_Date") >= start_bound)
            & (F.col("Posting_Date") <= end_bound)
        )
        .cache()
    )

# ---------- Work-order enrichment helpers ----------
def build_work_order_equipment_df(
    work_orders_df: DataFrame,
    lookup_tables: Dict[str, DataFrame],
) -> DataFrame:
    """Join work orders to equipment- and service-description lookups."""
    equipment_object_df, equipment_category_df, equipment_group_df, equipment_model_df, manufacturer_df, service_type_df = [
        lookup_tables[key]
        for key in [
            "Equipment_Object",
            "Equipment_Category",
            "Equipment_Group",
            "Equipment_Model",
            "Manufacturer",
            "Service_Type",
        ]
    ]
    return (
        work_orders_df.alias("wo")
        .join(
            broadcast(equipment_object_df.alias("equipment_object")),
            F.col("wo.Equipment_Object") == F.col("equipment_object.No_"),
            "left",
        )
        .join(
            broadcast(equipment_category_df.alias("equipment_category")),
            F.col("equipment_object.Equipment_Category")
            == F.col("equipment_category.Code"),
            "left",
        )
        .join(
            broadcast(equipment_group_df.alias("equipment_group")),
            F.col("equipment_object.Equipment_Group") == F.col("equipment_group.Code"),
            "left",
        )
        .join(
            broadcast(equipment_model_df.alias("equipment_model")),
            F.col("equipment_object.Equipment_Model") == F.col("equipment_model.No_"),
            "left",
        )
        .join(
            broadcast(manufacturer_df.alias("manufacturer")),
            F.col("equipment_object.Manufacturer_Code") == F.col("manufacturer.code"),
            "left",
        )
        .join(
            broadcast(service_type_df.alias("service_type")),
            F.col("wo.Service_Type") == F.col("service_type.Code"),
            "left",
        )
        .select(
            F.col("wo.No_").alias("Work_Order_No"),
            F.col("wo.Starting_Date"), F.col("wo.Posting_Date"),
            F.col("wo.Sell_to_Customer_No_").alias("Sell_to_Customer_No"),
            "wo.Responsibility_Center",
            F.concat_ws(": ",
                        F.col("service_type.Code"),
                        F.concat_ws(" / ",
                                    F.col("service_type.Description"),
                                    F.col("service_type.Service_Description")
                        )).alias("Service_Type_Description"),
            F.col("equipment_object.Description").alias("Equipment_Description"),
            F.col("equipment_category.Description").alias("Equipment_Category"),
            F.col("equipment_group.Description").alias("Equipment_Group"),
            F.col("equipment_model.Description").alias("Equipment_Model"),
            F.col("manufacturer.name").alias("Manufacturer"),
            "equipment_object.Year",
            F.round("wo.M1", 2).alias("Key_Hours"),
            F.round("wo.M2", 2).alias("Pump_Hours"),
            F.round("wo.M3", 2).alias("Drive_Hours"),
        )
    )

def process_rental_comment_lines(spark: SparkSession) -> DataFrame:
    """Assemble combined complaint/cause/cure/internal comments per work order."""
    raw_comments_df = (
        read_lakehouse_table(spark, "Rental_Comment_Line")
        .filter(F.col("Table_Type") == "11021629")
    )
    labeled_comments_df = raw_comments_df.withColumn(
        "DocLineType",
        F.when(F.col("Doc_Line") == 0, "Complaint")
         .when(F.col("Doc_Line") == 1, "Cause")
         .when(F.col("Doc_Line") == 2, "Cure_Comment")
         .when(F.col("Doc_Line") == 3, "Int_Comment")
    )
    sorted_comments_df = labeled_comments_df.groupBy("Doc_No_", "DocLineType").agg(
        F.sort_array(
            F.collect_list(F.struct("Subline","Comment_Line_No_","Comment")),
            asc=True).alias("sorted_comments")
    )
    concatenated_comments_df = sorted_comments_df.withColumn(
        "concatenated_comment",
        F.expr("array_join(transform(sorted_comments, x -> x.Comment), ' ')")
    )
    return (
        concatenated_comments_df.groupBy("Doc_No_")
        .pivot("DocLineType", ["Complaint","Cause","Cure_Comment","Int_Comment"])
        .agg(F.first("concatenated_comment"))
        .select("Doc_No_", "Complaint", "Cause", "Cure_Comment", "Int_Comment")
    )

def combine_work_orders_with_comments(
    work_order_equipment_df: DataFrame,
    rental_comments_df: DataFrame,
) -> DataFrame:
    """Join enriched work orders with the flattened rental comment text."""
    return (
        work_order_equipment_df.alias("wo")
        .join(
            rental_comments_df.alias("rc"),
            F.col("wo.Work_Order_No") == F.col("rc.Doc_No_"),
            "left",
        )
        .select(
            "wo.*",  # brings through all columns from info_df
            "rc.Complaint","rc.Cause","rc.Cure_Comment","rc.Int_Comment"
        )
    )

def find_parts_used(work_orders_df: DataFrame, lookup_tables: Dict[str, DataFrame]) -> DataFrame:
    """Aggregate invoiced parts per work order."""
    work_order_lines_df = lookup_tables["Work_Order_Line"]
    item_df = lookup_tables["Item"]
    item_category_df = lookup_tables["Item_Category"]
    return (
        work_orders_df.alias("work_order_header")
        .join(
            work_order_lines_df.alias("work_order_line"),
            F.col("work_order_header.No_") == F.col("work_order_line.Document_No_"),
            "inner",
        )
        .join(
            broadcast(item_df.alias("item")),
            F.col("work_order_line.No_") == F.col("item.No_"),
            "inner",
        )
        .join(
            broadcast(item_category_df.alias("item_category")),
            F.col("item.Item_Category_Code") == F.col("item_category.Code"),
            "left",
        )
        .groupBy("work_order_header.No_")
        .agg(
            F.round(F.sum("work_order_line.Amount_Invoiced"), 2).alias("Total_Amount_Invoiced"),
            F.array_join(
                F.collect_set(
                    F.concat_ws(
                        ", ",
                        F.col("item.Description"),
                        F.col("item.Manufacturer_Code"),
                        F.coalesce(F.col("item_category.Description"), F.lit(""))
                    )
                ),
                "; "
            ).alias("Parts_Used_Full")
        )
        .select(
            F.col("work_order_header.No_").alias("Work_Order_No"),
            "Parts_Used_Full",
            "Total_Amount_Invoiced",
        )
    )

def assemble_work_order_fact(
    commented_df: DataFrame,
    parts_summary_df: DataFrame,
) -> DataFrame:
    """Merge comment-enriched work orders with the parts usage summary."""
    return (
        commented_df.alias("comment_enriched")
        .join(parts_summary_df.alias("parts_usage"), "Work_Order_No", "left")
        .select(
            "comment_enriched.*",
            "parts_usage.Parts_Used_Full",
            "parts_usage.Total_Amount_Invoiced",
        )
    )

# ---------- Synapse utility helpers ----------
def align_with_synapse_schema(dataframe: DataFrame) -> DataFrame:
    """
    Cast columns and enforce ordering to match the Synapse `DE_LH_SILVER.dbo.fleet_ai` table schema.
    """
    missing_columns = [
        column for column in SYNAPSE_OUTPUT_COLUMNS if column not in dataframe.columns
    ]
    if missing_columns:
        raise ValueError(
            f"Missing expected column(s) for Synapse output: {', '.join(missing_columns)}"
        )

    aligned_df = dataframe
    for column_name, target_type in SYNAPSE_COLUMN_CASTS.items():
        if column_name not in aligned_df.columns:
            continue

        column_expr = F.col(column_name)
        if column_name in {"Pump_Hours", "Drive_Hours"}:
            column_expr = F.round(column_expr.cast("double")).cast(target_type)
        else:
            column_expr = column_expr.cast(target_type)

        aligned_df = aligned_df.withColumn(column_name, column_expr)

    return aligned_df.select(*SYNAPSE_OUTPUT_COLUMNS)


def get_existing_keys(
    spark: SparkSession,
    table_name: str,
    key_column: str,
    source_schema: StructType,
) -> Tuple[DataFrame, bool]:
    """Return existing key values for the target table, or an empty frame when missing."""
    key_field = next((field for field in source_schema if field.name == key_column), None)
    if key_field is None:
        raise ValueError(f"Column '{key_column}' was not found in the source schema.")

    try:
        existing = spark.read.table(table_name).select(key_column).distinct()
        return existing, True
    except Exception:
        empty_schema = StructType([StructField(key_column, key_field.dataType, key_field.nullable)])
        return spark.createDataFrame([], empty_schema), False


def append_new_records_to_delta(
    spark: SparkSession,
    dataset: DataFrame,
    table_name: str,
    key_column: str,
) -> int:
    """Append only new records to a Delta table, returning the count of inserted rows."""
    existing_keys, table_exists = get_existing_keys(spark, table_name, key_column, dataset.schema)
    new_records_df = (
        dataset.alias("incoming")
        .join(existing_keys.alias("existing"), key_column, "left_anti")
        .cache()
    )

    try:
        new_count = new_records_df.count()
        print(f"Number of new {table_name} records to append: {new_count}")

        if new_count == 0:
            print(f"No new records to append; {table_name} is already up-to-date.")
            return 0

        writer = new_records_df.write.format("delta")
        if table_exists:
            writer.option("mergeSchema", "true").mode("append").saveAsTable(table_name)
        else:
            writer.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)

        print(f"New records appended to {table_name}.")
        return new_count
    finally:
        new_records_df.unpersist()


def upsert_records_to_delta(
    spark: SparkSession,
    dataset: DataFrame,
    table_name: str,
    key_column: str,
) -> int:
    """Upsert records into a Delta table using the provided key column."""
    record_count = dataset.count()
    print(f"Preparing to upsert {record_count} record(s) into {table_name}.")

    if record_count == 0:
        print(f"No records supplied for upsert into {table_name}; skipping.")
        return 0

    if spark.catalog.tableExists(table_name):
        try:
            delta_table = DeltaTable.forName(spark, table_name)
            (
                delta_table.alias("target")
                .merge(
                    dataset.alias("source"),
                    f"target.{key_column} = source.{key_column}",
                )
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute()
            )
            print(f"Upsert completed for existing Delta table {table_name}.")
        except AnalysisException as err:
            raise RuntimeError(
                f"Failed to load Delta table {table_name} for upsert."
            ) from err
    else:
        (
            dataset.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )
        print(f"Delta table {table_name} created with {record_count} record(s).")

    return record_count


def count_duplicate_keys(spark: SparkSession, table_name: str, key_column: str) -> int:
    """Return the number of duplicate key values in the specified Delta table."""
    return (
        spark.read.table(table_name)
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )


def run_fleet_ai_etl(
    spark: SparkSession,
    start_date: str = DEFAULT_START_DATE,
    end_date: Optional[str] = None,
    load_mode: str = INCREMENTAL_MODE,
    *,
    key_column: str = KEY_COLUMN,
    target_table: str = DELTA_TARGET_TABLE,
) -> DataFrame:
    """Execute the Fleet AI ETL pipeline and return the assembled Spark DataFrame."""
    normalized_mode = load_mode.lower()
    if normalized_mode not in {INCREMENTAL_MODE, REPROCESS_MODE}:
        raise ValueError(
            f"Unsupported pipeline mode '{load_mode}'. "
            f"Expected '{INCREMENTAL_MODE}' or '{REPROCESS_MODE}'."
        )

    lookups = load_lookup_tables(spark)

    # Stage 1: pull base work orders and enrich with descriptive lookup data.
    work_orders_df = prepare_work_orders(spark, start_date, end_date)
    equipment_enriched_df = build_work_order_equipment_df(work_orders_df, lookups)
    # Stage 2: hydrate the textual context (comments and parts usage).
    rental_comments_df = process_rental_comment_lines(spark)
    commented_work_orders_df = combine_work_orders_with_comments(
        equipment_enriched_df,
        rental_comments_df,
    )
    parts_summary_df = find_parts_used(work_orders_df, lookups)
    final_df = assemble_work_order_fact(commented_work_orders_df, parts_summary_df).cache()

    # Stage 3: persist rows to the Delta target (append vs upsert) and report duplicates.
    if normalized_mode == REPROCESS_MODE:
        new_count = upsert_records_to_delta(
            spark,
            final_df,
            target_table,
            key_column,
        )
    else:
        new_count = append_new_records_to_delta(
            spark,
            final_df,
            target_table,
            key_column,
        )
    duplicate_count = count_duplicate_keys(
        spark,
        target_table,
        key_column,
    )

    if duplicate_count:
        print(f"WARNING: Detected {duplicate_count} duplicate {key_column}(s) in {target_table}!")
    else:
        print(f"No duplicates found in {target_table}.")

    work_orders_df.unpersist()
    return final_df


# ---------- Driver ----------
spark = init_spark()


def run_pipeline(
    start_date: Optional[str] = None,
    end_date: Optional[str] = None,
    mode: Optional[str] = None,
) -> DataFrame:
    """Convenience wrapper used by downstream notebook cells."""
    effective_start = start_date or PIPELINE_START_DATE
    effective_end = PIPELINE_END_DATE if end_date is None else end_date
    effective_mode = (mode or PIPELINE_MODE).lower()
    return run_fleet_ai_etl(
        spark,
        start_date=effective_start,
        end_date=effective_end,
        load_mode=effective_mode,
    )


final_df = run_pipeline()

if __name__ == "__main__":
    # Allow local execution with custom parameters if desired.
    run_pipeline()

StatementMeta(, c11a372b-28ee-44a1-bff7-11c9f6235f7f, 5, Finished, Available, Finished, False)

Preparing to upsert 2730 record(s) into DE_LH_SILVER.fleet_ai_master.
Upsert completed for existing Delta table DE_LH_SILVER.fleet_ai_master.
No duplicates found in DE_LH_SILVER.fleet_ai_master.
Preparing to upsert 2730 record(s) into DE_LH_SILVER.fleet_ai_master.
Upsert completed for existing Delta table DE_LH_SILVER.fleet_ai_master.
No duplicates found in DE_LH_SILVER.fleet_ai_master.


## Select Work Orders Missing from `fleet_ai`
This section finds the set of work orders that exist in lakehouse `fleet_ai_master` but are not yet in Synapse `DE_LH_SILVER.dbo.fleet_ai`. The result is stored as `pending_ai_records_df` and becomes the input to AI processing.
Steps: read distinct keys from Synapse, anti-join with the Delta table, cache and log the count.

In [2]:
# Identify Fleet AI records that exist in the lakehouse but are missing from Synapse,
# unless reprocess mode is active (in which case we stage the freshly built window).
is_reprocess_mode = PIPELINE_MODE.lower() == REPROCESS_MODE

if is_reprocess_mode:
    print("Reprocess mode enabled; staging configured window for AI enrichment.")
    pending_ai_records_df = final_df
else:
    target_existing_keys = (
        spark.read.table(LAKEHOUSE_TARGET_TABLE)
        .select(KEY_COLUMN)
        .distinct()
    )

    pending_ai_records_df = (
        spark.read.table(DELTA_TARGET_TABLE)
        .join(target_existing_keys, KEY_COLUMN, "left_anti")
        .cache()
    )

pending_ai_record_count = pending_ai_records_df.count()
print(f"Work orders queued for AI enrichment: {pending_ai_record_count}")

if pending_ai_record_count == 0:
    print(
        "No outstanding Fleet AI records detected; downstream AI processing will operate on an empty dataset."
    )


StatementMeta(, c11a372b-28ee-44a1-bff7-11c9f6235f7f, 6, Finished, Available, Finished, False)

Reprocess mode enabled; staging configured window for AI enrichment.
Work orders queued for AI enrichment: 2730


# AI Processing Overview
Input: `pending_ai_records_df` (work orders in `fleet_ai_master` that are not yet in Synapse `DE_LH_SILVER.dbo.fleet_ai`).
Steps:
- Generate `Service_Category` (classification).
- Generate `Damage_Category` (only for eligible service categories).
- Generate a customer-facing `Work_Order_Comment` (concise summary).
Outputs:
- `pandas_df`: AI-enriched rows in pandas for easier row-wise operations.
- `spark_df`: Spark conversion of `pandas_df` used for Synapse append.
Notes:
- Azure OpenAI credentials come from env vars: FABRIC_AZURE_OPENAI_KEY, FABRIC_AZURE_OPENAI_ENDPOINT, optional FABRIC_AZURE_OPENAI_API_VERSION, FABRIC_AZURE_OPENAI_DEPLOYMENT.
- Retries/backoff are enabled for robustness.

In [3]:
import os
from typing import Iterable, List, Optional

import pandas as pd
from tqdm.auto import tqdm
from openai import AzureOpenAI
from tenacity import retry, wait_exponential, stop_after_attempt

AZURE_OPENAI_KEY_ENV = "FABRIC_AZURE_OPENAI_KEY"
AZURE_OPENAI_ENDPOINT_ENV = "FABRIC_AZURE_OPENAI_ENDPOINT"
AZURE_OPENAI_VERSION_ENV = "FABRIC_AZURE_OPENAI_API_VERSION"
AZURE_OPENAI_DEPLOYMENT_ENV = "FABRIC_AZURE_OPENAI_DEPLOYMENT"

AZURE_OPENAI_KEY_LITERAL = "33d0519dcebb44688bdfa776b6810d81"
AZURE_OPENAI_ENDPOINT_LITERAL = "https://fabricazureopenaiprolift.openai.azure.com/"
AZURE_OPENAI_API_VERSION_LITERAL = "2024-10-21"
AZURE_OPENAI_DEPLOYMENT_LITERAL = "gpt-4o-mini"

DEFAULT_AZURE_OPENAI_API_VERSION = os.getenv(
    AZURE_OPENAI_VERSION_ENV,
    AZURE_OPENAI_API_VERSION_LITERAL,
)
DEFAULT_AZURE_OPENAI_DEPLOYMENT = os.getenv(
    AZURE_OPENAI_DEPLOYMENT_ENV,
    AZURE_OPENAI_DEPLOYMENT_LITERAL,
)

_azure_openai_client: Optional[AzureOpenAI] = None


def get_azure_openai_client() -> AzureOpenAI:
    """Create (or reuse) a single Azure OpenAI client based on environment / secret scope variables."""
    global _azure_openai_client

    if _azure_openai_client is None:
        api_key = os.getenv(AZURE_OPENAI_KEY_ENV, AZURE_OPENAI_KEY_LITERAL)
        endpoint = os.getenv(AZURE_OPENAI_ENDPOINT_ENV, AZURE_OPENAI_ENDPOINT_LITERAL)
        if not api_key or not endpoint:
            raise RuntimeError(
                "Azure OpenAI credentials are not configured. "
                f"Set both {AZURE_OPENAI_KEY_ENV} and {AZURE_OPENAI_ENDPOINT_ENV} via environment variables or notebook secrets."
            )

        api_version = DEFAULT_AZURE_OPENAI_API_VERSION
        _azure_openai_client = AzureOpenAI(
            api_key=api_key,
            api_version=api_version,
            azure_endpoint=endpoint,
        )

    return _azure_openai_client


def invoke_chat_completion(
    messages: List[dict],
    *,
    temperature: float,
    max_tokens: int,
    default_value: str,
) -> str:
    """Execute a chat completion call and fall back gracefully when the API errors or returns no text."""
    try:
        response = get_azure_openai_client().chat.completions.create(
            model=DEFAULT_AZURE_OPENAI_DEPLOYMENT,
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        content = (response.choices[0].message.content or "").strip()
        return content or default_value
    except Exception as exc:
        # Surface the failure for observability but continue processing downstream rows.
        print(f"Azure OpenAI invocation failed: {exc}")
        return default_value


def contains_keyword(text: str, keywords: Iterable[str]) -> bool:
    """Case-insensitive containment check against a keyword list."""
    lowered = (text or "").lower()
    return any(keyword.lower() in lowered for keyword in keywords)


PLANNED_MAINTENANCE_KEYWORDS: tuple[str, ...] = (
    "CBRONZEPM",
    "CDPM",
    "CPMTM",
    "CPMTP",
    "CTMMIPM",
    "CTMMKPM",
    "CTMMWVPM",
    "FLEXRENTPM",
    "MGOLDPM",
    "MPLATPM",
    "MSILVERPM",
    "MSILVERPM+",
    "PREPAID PM",
    "RERENTPM",
    "RPM",
    "RPO PM",
    "T360PM",
    "USEDRENTPM",
    "WECNPM",
    "WECPM",
)

INTERNAL_SERVICE_KEYWORDS: tuple[str, ...] = (
    "5S - ASEC",
    "CTPDI",
    "DOCK-INT",
    "FLEXRENTAL",
    "FLEXRENTPM",
    "MGOLD",
    "MGOLDPM",
    "MSILVERPM",
    "MSILVERPM+",
    "NAFTERS",
    "NAFTERSTOY",
    "NDEMO",
    "NINVTRF",
    "NLOAD",
    "NPDI",
    "NPDIF",
    "NPDIPARTS",
    "NPDITOY",
    "NPDITOYF",
    "OFFLEASE",
    "PLABOR",
    "PRETURNS",
    "PTRUCK",
    "PWARR",
    "REMAN-BILD",
    "REMAN-PULL",
    "RERENTPM",
    "RGM",
    "RPM",
    "RPO",
    "RPO PM",
    "RRENT",
    "RRERENT",
    "SCOMMIT",
    "SDCLND",
    "SDEL",
    "SRETURNS",
    "STOTALCARE",
    "STRUCK",
    "STRUCKDEL",
    "SVANINV",
    "SVC SUPERV",
    "SWARR",
    "T360PM",
    "TECH",
    "TIRE-GOLD",
    "TIRE-RENT",
    "TIRE-USED",
    "UAFTERS",
    "UDEMO",
    "ULOAD",
    "USED-RECON",
    "USEDRENTAL",
    "USEDRENTPM",
    "UWARR",
    "WARCHEST",
    "WECNPM",
    "WECPM",
    "WMFG",
    "WOTHERMFG",
)

DAMAGE_ELIGIBLE_SERVICE_CATEGORIES: tuple[str, ...] = (
    "Damage",
    "Tires",
    "Battery / Charger",
    "Accessory Repair",
    "Attachment Repair",
)

COMMENT_MAX_TOKENS = 100
CLASSIFICATION_MAX_TOKENS = 20

StatementMeta(, c11a372b-28ee-44a1-bff7-11c9f6235f7f, 7, Finished, Available, Finished, False)

### Generate Work Order Comment

In [4]:
@retry(wait=wait_exponential(min=1, max=60), stop=stop_after_attempt(6))
def generate_work_order_comment(
    work_order_no,
    service_type_description,
    equipment_description,
    equipment_category,
    equipment_group,
    equipment_model,
    manufacturer,
    year,
    key_hours,
    pump_hours,
    drive_hours,
    complaint,
    cause,
    cure_comment,
    int_comment,
    parts_used_full
):
    """Generate a concise, customer-facing summary comment for the work order."""
    if contains_keyword(service_type_description, PLANNED_MAINTENANCE_KEYWORDS):
        return "Planned Maintenance"

    system_message = """
You are an AI assistant that generates short, customer-facing Work Order comments.
Based on the forklift work order data, follow these rules:

1. If there are no comments, create a concise comment describing the work or equipment details.
2. If there are any comments, combine them (Complaint, Cause, Cure, Int) into a single 
    short paragraph for the customer. Remove internal jargon and keep it simple.
3. Comments should always be past tense.

Only provide the final comment. Do not include extra explanations or disclaimers.
    """

    user_message = f"""
Below is the Work Order Data to classify:

Work Order No: {work_order_no}
Service Type Description: {service_type_description}
Equipment Description: {equipment_description}
Equipment Category: {equipment_category}
Equipment Group: {equipment_group}
Equipment Model: {equipment_model}
Manufacturer: {manufacturer}
Year: {year}
Key Hours: {key_hours}
Pump Hours: {pump_hours}
Drive Hours: {drive_hours}

Comments:
- Complaint: {complaint}
- Cause: {cause}
- Cure Comment: {cure_comment}
- Int Comment: {int_comment}

Parts Used:
{parts_used_full}
    """

    messages = [
        {"role": "system", "content": system_message.strip()},
        {"role": "user", "content": user_message.strip()},
    ]

    return invoke_chat_completion(
        messages,
        temperature=0.5,
        max_tokens=COMMENT_MAX_TOKENS,
        default_value="Unable to generate comment.",
    )


StatementMeta(, c11a372b-28ee-44a1-bff7-11c9f6235f7f, 8, Finished, Available, Finished, False)

### Classify Service Type

In [5]:
@retry(wait=wait_exponential(min=1, max=60), stop=stop_after_attempt(6))
def classify_service_type(
    work_order_no,
    service_type_description,
    equipment_description,
    equipment_category,
    equipment_group,
    equipment_model,
    manufacturer,
    year,
    key_hours,
    pump_hours,
    drive_hours,
    complaint,
    cause,
    cure_comment,
    int_comment,
    parts_used_full,
    total_amount_invoiced
):

    if contains_keyword(service_type_description, PLANNED_MAINTENANCE_KEYWORDS):
        return "PM Service"
    if contains_keyword(service_type_description, INTERNAL_SERVICE_KEYWORDS):
        return "Internal"
    
    elif total_amount_invoiced is None or total_amount_invoiced == 0:
        return "Non-Billable / No Charge"

    system_prompt = """
You are classifying forklift and related equipment service work orders into exactly one 
non-human, non-harmful category. This task involves only machinery, components, and 
routine service descriptions. No part of this task involves people, injuries, unsafe 
actions, or harmful behavior. All references to “damage” refer strictly to equipment 
conditions.

Your output should be exactly one of the following category names:
"Telemetry", "Battery Watering", "Battery / Charger", "Accessory Repair", 
"Attachment Repair", "Tires", "Seats", "General Repair", "Damage", or "Research".

=== Classification Logic ===

0) Equipment Type Considerations:
- If the equipment is labeled “IC” (internal combustion), do not use Battery / Charger 
  even if battery-related words appear. Use General Repair unless another rule 
  clearly indicates a different category.
- If the equipment type is “Battery, Chargers...” and the work involves battery or 
  charger components, classify as Battery / Charger.
- For scrubbers or floor machines: do not use Battery Watering or Attachment Repair. 
  If the machine is clearly electric and the issue involves the battery or charger, 
  use Battery / Charger.

1) Service Type Restrictions:
If the service type is one of the following:
CADDL-DAM, CDAMAGE, CDAMAGE-F, CTMMIPRXAD, CTMMIVMSAD, MPLATAD, TIREDAMAGE
→ The category must be one of: Accessory Repair, Attachment Repair, Tires, 
  Battery / Charger, or Damage.

2) Equipment Damage Indicators:
The following are neutral text cues describing mechanical conditions, not human harm:
“hit”, “bent”, “damaged”, “impact”, “torn”, “broken”, “cracked”, 
“knocked”, “fallen off”, “wrapped around”.

If these cues appear, classify as Damage unless:
- The issue is a normal-wear item without signs of external force 
  (e.g., floor mat, wiper).
- An attachment component is affected → Attachment Repair.
- A seat is torn or broken → Damage.
- A heater issue is from gradual wear → Accessory Repair; if from impact → Damage.

3) Keyword-Based Category Selection (first applicable match):
1. Telemetry: “VMS”, “Vac4”, “T-Matics”, “key troller” (fleet tracking/operator login).
2. Battery Watering: “battery watering”, “water level”, or work primarily involving adding water.
3. Battery / Charger: “battery”, “cell”, “charger” 
   (only for electric or battery-labeled equipment; if “IC” is present, use General Repair).
4. Accessory Repair: computer, tablet, scanner, radio, optional accessories, 
   or gradual heater wear.
5. Attachment Repair: side shifter, clamp, push/pull, attachment hoses or fittings.
6. Tires: tires, load wheels, bearings.
7. Seats: seat-related issues (not involving tearing or breakage).
8. General Repair: standard mechanical/electrical issues (horns, alternators, sensors), 
   normal wear items (strobe light, floor mats), fueling, fans (engine/radiator), 
   primary joystick on lifts, and similar routine repairs.

4) Additional Clarifications:
- If details conflict or information is insufficient, classify as Research.
- If no keyword match appears and the repair is routine, use General Repair.
- Mirrors: new installation → Accessory Repair; damaged mirror → Damage.
- Operator cooling fan → Accessory Repair; radiator/engine fan → General Repair.
- Hoses not part of an attachment → General Repair; attachment hoses → Attachment Repair.

Return only the single category name.
"""

    user_prompt = f"""
Below is the Work Order Data to classify:

Work Order No: {work_order_no}
Service Type Description: {service_type_description}
Equipment Description: {equipment_description}
Equipment Category: {equipment_category}
Equipment Group: {equipment_group}
Equipment Model: {equipment_model}
Manufacturer: {manufacturer}
Year: {year}
Key Hours: {key_hours}
Pump Hours: {pump_hours}
Drive Hours: {drive_hours}
Total_Amount_Invoiced: {total_amount_invoiced}

Comments:
- Complaint: {complaint}
- Cause: {cause}
- Cure Comment: {cure_comment}
- Int Comment: {int_comment}

Parts Used:
{parts_used_full}
"""

    messages = [
        {"role": "system", "content": system_prompt.strip()},
        {"role": "user", "content": user_prompt.strip()},
    ]

    category = invoke_chat_completion(
        messages,
        temperature=0.0,
        max_tokens=CLASSIFICATION_MAX_TOKENS,
        default_value="No category found.",
    )

    return category if "\n" not in category else "No category found."

StatementMeta(, c11a372b-28ee-44a1-bff7-11c9f6235f7f, 9, Finished, Available, Finished, False)

### Classify Equipment Damage

In [6]:
@retry(wait=wait_exponential(min=1, max=60), stop=stop_after_attempt(6))
def classify_damage_type(
    work_order_no,
    service_type_description,
    equipment_description,
    equipment_category,
    equipment_group,
    equipment_model,
    manufacturer,
    year,
    key_hours,
    pump_hours,
    drive_hours,
    complaint,
    cause,
    cure_comment,
    int_comment,
    parts_used_full
):

    system_message = """
You are performing a routine text-classification task for forklift service documentation. 
This task involves only equipment, machinery parts, and maintenance descriptions. 
There is no content related to people, injuries, harmful actions, or unsafe behavior. 
All references to “damage” refer strictly to non-human, mechanical components.

Objective:
Determine whether a work-order description suggests externally caused physical damage 
to forklift parts, or whether it reflects normal wear, routine maintenance, or 
non-damage-related modifications. If no externally caused damage is indicated, 
the appropriate category is “No Damage.”

Categories:
1. Attachment Damage: issues involving attachments or lifting mechanisms 
   (sideshifter, clamp, forks, mast, hoses, bolts, chains, cylinders)
2. Tire Damage: externally caused tire or wheel issues 
   (puncture, debris, flat, leak)
3. Battery Damage: battery-related components 
   (battery, charger, cables, connector)
4. Rental Damage: issues specifically tied to rental use
5. Dock Damage: issues associated with dock areas
6. Other Damage: other forklift components 
   (frame, door, counterweight, steering, seat, cushions, arm rests, handles)
7. Accessory Damage: add-on components 
   (lights, mirrors, scanners, wiring)
8. No Damage: normal wear, routine maintenance, or modifications without 
   evidence of external damage

Text cues that may suggest equipment damage (these are neutral descriptors of 
mechanical conditions, not related to people): 
“hit”, “bent”, “damaged”, “impact”, “torn”, “accident”, “puncture”, 
“broken”, “leak”, “cut”, “missing”.

Guidelines:
- Interpret the work order as a description of equipment condition only.
- Use “No Damage” when the text does not indicate externally caused damage.
- If a component is described with terms such as broken, bent, cut, or missing, 
  treat this as evidence of equipment damage.
- Tire Damage should be used only when there is a clear external cause.
- Provide only the single most appropriate category name as the final output.
    """

    user_message = f"""
Below is the forklift work order data for Work Order #{work_order_no}.

Data:
Work Order No: {work_order_no}
Service Type Description: {service_type_description}
Equipment Description: {equipment_description}
Equipment Category: {equipment_category}
Equipment Group: {equipment_group}
Equipment Model: {equipment_model}
Manufacturer: {manufacturer}
Year: {year}
Key Hours: {key_hours}
Pump Hours: {pump_hours}
Drive Hours: {drive_hours}
Complaint: {complaint}
Cause: {cause}
Cure Comment: {cure_comment}
Int Comment: {int_comment}
Parts Used Full: {parts_used_full}
    """

    messages = [
        {"role": "system", "content": system_message.strip()},
        {"role": "user", "content": user_message.strip()},
    ]

    return invoke_chat_completion(
        messages,
        temperature=0.0,
        max_tokens=CLASSIFICATION_MAX_TOKENS,
        default_value="Filtered",
    )


StatementMeta(, c11a372b-28ee-44a1-bff7-11c9f6235f7f, 10, Finished, Available, Finished, False)

### Apply AI Functions to `pending_ai_records_df`
- Applies: service classification, damage classification (eligible only), and comment generation.
- Produces three new columns: `Service_Category`, `Damage_Category`, `Work_Order_Comment`.
- Output: `pandas_df` (then converted to `spark_df` for writing).

In [7]:
if "pending_ai_records_df" not in globals():
    raise NameError(
        "pending_ai_records_df is not defined. Run the ETL cell above before executing AI processing."
    )


def enrich_with_ai_annotations(dataframe: DataFrame) -> pd.DataFrame:
    """Apply service classification, damage classification, and comment generation."""
    pdf = dataframe.toPandas()

    if pdf.empty:
        pdf["Service_Category"] = pd.Series(dtype="object")
        pdf["Damage_Category"] = pd.Series(dtype="object")
        pdf["Work_Order_Comment"] = pd.Series(dtype="object")
        return pdf

    def _classify_service(row):
        return classify_service_type(
            row["Work_Order_No"],
            row["Service_Type_Description"],
            row["Equipment_Description"],
            row["Equipment_Category"],
            row["Equipment_Group"],
            row["Equipment_Model"],
            row["Manufacturer"],
            row["Year"],
            row["Key_Hours"],
            row["Pump_Hours"],
            row["Drive_Hours"],
            row["Complaint"],
            row["Cause"],
            row["Cure_Comment"],
            row["Int_Comment"],
            row["Parts_Used_Full"],
            row["Total_Amount_Invoiced"],
        )

    def _classify_damage(row, service_category):
        if service_category not in DAMAGE_ELIGIBLE_SERVICE_CATEGORIES:
            return "No Damage"
        return classify_damage_type(
            row["Work_Order_No"],
            row["Service_Type_Description"],
            row["Equipment_Description"],
            row["Equipment_Category"],
            row["Equipment_Group"],
            row["Equipment_Model"],
            row["Manufacturer"],
            row["Year"],
            row["Key_Hours"],
            row["Pump_Hours"],
            row["Drive_Hours"],
            row["Complaint"],
            row["Cause"],
            row["Cure_Comment"],
            row["Int_Comment"],
            row["Parts_Used_Full"],
        )

    def _generate_comment(row):
        return generate_work_order_comment(
            row["Work_Order_No"],
            row["Service_Type_Description"],
            row["Equipment_Description"],
            row["Equipment_Category"],
            row["Equipment_Group"],
            row["Equipment_Model"],
            row["Manufacturer"],
            row["Year"],
            row["Key_Hours"],
            row["Pump_Hours"],
            row["Drive_Hours"],
            row["Complaint"],
            row["Cause"],
            row["Cure_Comment"],
            row["Int_Comment"],
            row["Parts_Used_Full"],
        )

    total_rows = len(pdf.index)
    service_categories: List[str] = []
    damage_categories: List[str] = []
    work_order_comments: List[str] = []
    progress_bar = (
        tqdm(total=total_rows, desc="AI enrichment", unit="order") if total_rows else None
    )

    for _, row in pdf.iterrows():
        service_category = _classify_service(row)
        damage_category = _classify_damage(row, service_category)
        work_order_comment = _generate_comment(row)

        service_categories.append(service_category)
        damage_categories.append(damage_category)
        work_order_comments.append(work_order_comment)

        if progress_bar:
            progress_bar.update(1)

    if progress_bar:
        progress_bar.close()

    pdf["Service_Category"] = service_categories
    pdf["Damage_Category"] = damage_categories
    pdf["Work_Order_Comment"] = work_order_comments

    if total_rows:
        tqdm.write(f"AI enrichment complete: {total_rows} work order(s) processed.")
    return pdf


pandas_df = enrich_with_ai_annotations(pending_ai_records_df)
pending_ai_records_df.unpersist()

StatementMeta(, c11a372b-28ee-44a1-bff7-11c9f6235f7f, 11, Finished, Available, Finished, False)

AI enrichment:   0%|          | 0/2730 [00:00<?, ?order/s]

AI enrichment complete: 2730 work order(s) processed.


DataFrame[Work_Order_No: string, Starting_Date: timestamp, Posting_Date: timestamp, Sell_to_Customer_No: string, Responsibility_Center: string, Service_Type_Description: string, Equipment_Description: string, Equipment_Category: string, Equipment_Group: string, Equipment_Model: string, Manufacturer: string, Year: int, Key_Hours: decimal(21,2), Pump_Hours: decimal(21,2), Drive_Hours: decimal(21,2), Complaint: string, Cause: string, Cure_Comment: string, Int_Comment: string, Parts_Used_Full: string, Total_Amount_Invoiced: decimal(21,2)]

StatementMeta(, c11a372b-28ee-44a1-bff7-11c9f6235f7f, 13, Finished, Available, Finished, False)

StatementMeta(, , -1, Finished, , Finished, True)

RejectSilentExecuteRequest: Livy session has failed. Error code: RejectSilentExecuteRequest. Rejected silent execute_request as there is no active session.

# Write Results to Lakehouse (mode-aware)
- Incremental: append only records whose `Work_Order_No` is not already present.
- Reprocess: upsert the staged window into the target, overwriting matching keys.
- Validation: logs rows affected and checks for duplicate keys after write.

In [8]:
import pyspark.sql.functions as F


def _target_table_identifier() -> str:
    """Return the fully qualified identifier for the Lakehouse target table."""
    return LAKEHOUSE_TARGET_TABLE


def _empty_target_dataframe(key_column: str) -> DataFrame:
    """Return an empty DataFrame with the expected key column when the target does not exist."""
    schema = StructType([StructField(key_column, StringType(), True)])
    return spark.createDataFrame([], schema)


def _read_target_table(key_column: str) -> DataFrame:
    """Load the Lakehouse target table; fallback to an empty frame if missing."""
    if spark.catalog.tableExists(LAKEHOUSE_TARGET_TABLE):
        return spark.read.table(LAKEHOUSE_TARGET_TABLE)
    print(f"Lakehouse table {LAKEHOUSE_TARGET_TABLE} not found; assuming no existing target rows.")
    return _empty_target_dataframe(key_column)


def append_new_records_to_target(
    dataset: DataFrame,
    *,
    key_column: str,
) -> int:
    """Append only new records to the Lakehouse target, returning the number of rows inserted."""
    target_identifier = _target_table_identifier()
    existing_keys = _read_target_table(key_column).select(key_column).distinct()
    new_records_df = (
        dataset.alias("incoming")
        .join(existing_keys.alias("existing"), key_column, "left_anti")
        .cache()
    )

    try:
        new_count = new_records_df.count()
        print(f"Number of new {target_identifier} records to append: {new_count}")

        if new_count == 0:
            print(f"No new records to append; {target_identifier} is already up-to-date.")
            return 0

        (
            new_records_df.write.format("delta")
            .mode("append")
            .saveAsTable(LAKEHOUSE_TARGET_TABLE)
        )
        print(f"New records appended to {target_identifier}.")
        return new_count
    finally:
        new_records_df.unpersist()


def persist_records_to_target(
    dataset: DataFrame,
    *,
    key_column: str,
    mode: str,
) -> int:
    """Persist records into the Lakehouse target using append or upsert semantics."""
    normalized_mode = mode.lower()
    if normalized_mode == REPROCESS_MODE:
        return upsert_records_to_delta(
            spark,
            dataset,
            LAKEHOUSE_TARGET_TABLE,
            key_column,
        )

    return append_new_records_to_target(
        dataset,
        key_column=key_column,
    )


def count_target_duplicates(key_column: str) -> int:
    """Return the number of duplicate rows in the Lakehouse target based on the key column."""
    return (
        _read_target_table(key_column)
        .groupBy(key_column)
        .agg(F.count("*").alias("record_count"))
        .filter(F.col("record_count") > 1)
        .count()
    )


# Convert pandas DataFrame to Spark DataFrame for downstream persistence.
if pandas_df.empty:
    print("No work orders to write to the configured target; skipping write operation.")
else:
    spark_df = spark.createDataFrame(pandas_df)
    target_ready_df = align_with_synapse_schema(spark_df)

    persist_records_to_target(
        target_ready_df,
        key_column=KEY_COLUMN,
        mode=PIPELINE_MODE,
    )

target_duplicates = count_target_duplicates(
    key_column=KEY_COLUMN,
)

target_identifier = _target_table_identifier()

if target_duplicates:
    print(f"WARNING: Detected {target_duplicates} duplicate {KEY_COLUMN}(s) in {target_identifier}!")
else:
    print(f"No duplicates found in {target_identifier}.")


StatementMeta(, c11a372b-28ee-44a1-bff7-11c9f6235f7f, 12, Finished, Available, Finished, False)

Preparing to upsert 2730 record(s) into DE_LH_SILVER.fleet_ai.
Upsert completed for existing Delta table DE_LH_SILVER.fleet_ai.
No duplicates found in DE_LH_SILVER.fleet_ai.
